In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy
from other_methods import delves_lyness, kravanja_van_barel
from tqdm import tqdm

from skzeros import Rectangle, find_zeros
from skzeros._domain import _subdivide_domain, get_leaf_regions

plt.style.use("style.mplstyle")

In [ ]:
# https://discourse.julialang.org/t/computing-the-adjugate-cofactors-of-a-matrix/125184
def adjugate(A):
    U, S, VH = np.linalg.svd(A)
    phase_det = np.linalg.det(VH @ U)
    return VH.T.conj() @ (_adjugate_diagonal(S) * phase_det) @ U.conj().T


def _adjugate_diagonal(d):
    n = len(d)
    if n < 2:
        return np.diag(np.ones_like(d))

    dadj = np.empty_like(d)
    prefix = 1
    for i in range(n):
        dadj[i] = prefix
        prefix *= d[i]

    suffix = 1
    for i in reversed(range(n)):
        dadj[i] *= suffix
        suffix *= d[i]

    return np.diag(dadj)

In [ ]:
rng = np.random.default_rng(1212)
n = 50
x = -0.4 + 0.8 * rng.integers(0, 1, n, endpoint=True)
A = scipy.linalg.circulant(x)

In [ ]:
e = np.linalg.eigvals(A)

In [ ]:
count = 0


@np.vectorize
def f(z):
    global count
    count += 1
    return np.linalg.det(A - z * np.eye(n))


@np.vectorize
def f_z(z):
    return -np.trace(adjugate(A - z * np.eye(n)))

In [ ]:
count = 0
d = Rectangle(complex(-5.1, -4.9), complex(5, 4.7))
r = find_zeros(d, f, f_z, 7.1, rng=rng)

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(4, 2))
d.plot(ax1)
ax1.plot(r.zeros.real, r.zeros.imag, ".")
ax1.set_xlabel(r"$\Re(z)$")
ax1.set_ylabel(r"$\Im(z)$")
ax1.set_aspect("equal", "box")
ax2.hist(np.log10(np.min(np.abs(np.subtract.outer(e, r.zeros)), axis=-1)))
ax2.set_xlim([-16, -5])
ax2.set_xlabel("Log Error")
ax2.set_ylabel("Frequency")
plt.tight_layout()
plt.savefig("figures/example3a.pdf")

In [ ]:
AAA_err = np.min(np.abs(np.subtract.outer(e, r.zeros)), axis=-1)
print(
    f"AAA: {count=}, max error={np.max(AAA_err):.2e}, min error={np.min(AAA_err):.2e}, medium error={np.median(AAA_err):.2e}"
)
print(len(get_leaf_regions(d)))

In [ ]:
count = 0
d = Rectangle(complex(-5.1, -4.9), complex(5, 4.7))
delves_lyness_zeros = []
regions = _subdivide_domain(
    d,
    f,
    f_z,
    max_arg_principle=7.1,
    maxiter=100,
    quadrature_args={"epsabs": 1e-3, "epsrel": 1e-3},
    rng=rng,
)
for region in tqdm(regions):
    # Get the expected number of zeros
    N = region._arg_principle
    # if there are no zeros then there is no work to be done!
    if np.isclose(N, 0, atol=1e-3):
        continue
    delves_lyness_zeros.append(
        delves_lyness(
            f,
            f_z,
            region,
            N=np.round(N.real).astype(int),
            quad_kwargs={"epsabs": 1e-12, "epsrel": 1e-12},
        )
    )

delves_lyness_zeros = np.concatenate(delves_lyness_zeros)

delves_lyness_err = np.min(np.abs(np.subtract.outer(e, delves_lyness_zeros)), axis=-1)
delves_lyness_count = count
print(
    f"Delves-Lyness: {count=}, max error={np.max(delves_lyness_err):.1e}, min error={np.min(delves_lyness_err):.1e}, medium error={np.median(delves_lyness_err):.1e}"
)
print(len(get_leaf_regions(d)))

In [ ]:
count = 0
kvb_zeros = []
d = Rectangle(complex(-5.1, -4.9), complex(5, 4.7))
regions = _subdivide_domain(
    d,
    f,
    f_z,
    max_arg_principle=3.1,
    maxiter=200,
    quadrature_args={"epsabs": 1e-1, "epsrel": 1e-1},
    rng=rng,
)
for region in tqdm(regions):
    # Get the expected number of zeros
    N = region._arg_principle
    # if there are no zeros then there is no work to be done!
    if np.isclose(N, 0, atol=1e-3):
        continue
    kvb_zeros.append(
        kravanja_van_barel(
            f,
            f_z,
            region,
            N=np.round(N.real).astype(int),
            quad_kwargs={"epsabs": 1e-12, "epsrel": 1e-12},
        )[0]
    )

kvb_zeros = np.concatenate(kvb_zeros)

kvb_err = np.min(np.abs(np.subtract.outer(e, kvb_zeros)), axis=0)
kvb_count = count
print(
    f"kvb: {count=}, max error={np.max(kvb_err):.1e}, min error={np.min(kvb_err):.1e}, medium error={np.median(kvb_err):.1e}"
)
print(len(get_leaf_regions(d)))

In [ ]:
count = 0
from cxroots import Rectangle as cxRectangle
from cxroots.root_approximation import approximate_roots

kvb_fop_zeros = []
d = Rectangle(complex(-5.1, -4.9), complex(5, 4.7))
regions = _subdivide_domain(
    d,
    f,
    f_z,
    max_arg_principle=4.1,
    maxiter=200,
    quadrature_args={"epsabs": 1e-1, "epsrel": 1e-1},
    rng=rng,
)
for region in tqdm(regions):
    # Get the expected number of zeros
    N = region._arg_principle
    # if there are no zeros then there is no work to be done!
    if np.isclose(N, 0, atol=1e-3):
        continue
    c = cxRectangle(
        (region.bottom_left.real, region.top_right.real),
        (region.bottom_left.imag, region.top_right.imag),
    )
    result = approximate_roots(
        c, np.round(N.real).astype(int), f, f_z, abs_tol=1e-12, rel_tol=1e-12
    )
    kvb_fop_zeros.append(result[0])

kvb_fop_zeros = np.concatenate(kvb_fop_zeros)
kvb_err = np.min(np.abs(np.subtract.outer(e, kvb_fop_zeros)), axis=0)
kvb_count = count
print(
    f"kvb_fop: {count=}, max error={np.max(kvb_err):.1e}, min error={np.min(kvb_err):.1e}, medium error={np.median(kvb_err):.1e}"
)
print(len(get_leaf_regions(d)))